# 5. The test suite

The repository's tests, run in this kernel. In the browser the test dependencies are installed at the versions recorded in `uv.lock`, read from the file itself. pydantic is the exception: its compiled core has to match the Pyodide build, so it comes from the Pyodide distribution.

Three markers name a capability a WebAssembly runtime may lack, and `tests/conftest.py` skips a marked test where the capability is missing:

| Marker | Tests | Skipped where |
| --- | ---: | --- |
| `threads` | 18 | the runtime cannot start a thread: Pyodide |
| `subprocess` | 1 | the runtime has no processes: Pyodide |
| `event_loop` | 9 | synchronous code cannot run an event loop: Pyodide in an engine without JSPI, and a desktop notebook, whose kernel is already running one |

Chromium and Firefox both support JSPI, so in either browser the 19 `threads` and `subprocess` tests are skipped and the `event_loop` tests run. An engine without JSPI skips all 28. On the desktop this notebook skips the 9 `event_loop` tests, and `make test` runs them outside a notebook. Notebook 3 runs the asyncio scenarios with top-level `await`, which needs neither threads nor JSPI.

In [ ]:
import sys
import tomllib
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pytest.ini').exists())
locked = {package['name']: package['version'] for package in tomllib.loads((root / 'uv.lock').read_text())['package']}
pinned = [
    f'{name}=={locked[name]}'
    for name in ('pytest', 'pytest-bdd', 'gherkin-official', 'factory-boy', 'faker', 'sqlalchemy', 'attrs')
]

if sys.platform == 'emscripten':  # in the browser: this repository's wheel, served with the site
    import piplite

    await piplite.install(['banking', *pinned, 'pydantic'])

print('Python', sys.version.split()[0], 'on', sys.platform)
print('pinned from uv.lock:', ', '.join(pinned))

In [ ]:
import os

import pytest

sys.dont_write_bytecode = True  # keeps __pycache__ out of the notebook's file browser
os.chdir(root)
exit_code = pytest.main(['tests', '-p', 'no:cacheprovider', '-q', '-rs'])
print('pytest exit code:', int(exit_code))